In [1]:
import os

In [2]:
%pwd

'd:\\DataScienceEnd-to-Endprojects\\Data_Science_LifeCycle\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\DataScienceEnd-to-Endprojects\\Data_Science_LifeCycle'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [6]:
from src.DataScience_lifecycle.constants import *
from src.DataScience_lifecycle.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(self, config_file_path=CONFIG_FILE_PATH, 
                    params_file_path=PARAMS_FILE_PATH, 
                    schema_file_path=SCHEMA_FILE_PATH):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)
        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])
        data_ingestion_config = DataIngestionConfig(root_dir=config.root_dir,
                                                    source_URL=config.source_URL,
                                                    local_data_file=config.local_data_file,
                                                    unzip_dir=config.unzip_dir)
        return data_ingestion_config

In [8]:
import requests
from src.DataScience_lifecycle import logger
import zipfile

In [ ]:
# component 1: data ingestion
class DataIngestion:
    def __init__(self,config:DataIngestionConfig):
        self.config = config

    #Dowloading zip file from source url and save it in local directory
    def download_file(self):
        os.makedirs(os.path.dirname(self.config.local_data_file), exist_ok=True)

        try:
            logger.info("Starting file download...")
            # Download file
            response = requests.get(self.config.source_URL)
            response.raise_for_status()

            # Save file
            with open(self.config.local_data_file, "wb") as file:
                file.write(response.content)

            file_size = os.path.getsize(self.config.local_data_file)

            logger.info(f"File downloaded successfully!")
            logger.info(f"File location: {self.config.local_data_file}")
            logger.info(f"File size: {file_size} bytes")

        except Exception as e:
            logger.error(f"Error while downloading file: {e}")
            raise e

    # Extract zip file
    def extract_zip_file(self):

        unzip_path = self.config.unzip_dir

        # Create unzip directory
        os.makedirs(unzip_path, exist_ok=True)

        if not os.path.exists(self.config.local_data_file):

            logger.error(
                f"File not found: {self.config.local_data_file}"
            )

            raise FileNotFoundError(
                f"File not found: {self.config.local_data_file}"
            )

        # Validate ZIP file
        if not zipfile.is_zipfile(self.config.local_data_file):

            logger.error(
                f"{self.config.local_data_file} is not a valid ZIP file"
            )

            raise ValueError(
                f"{self.config.local_data_file} is not a valid ZIP file"
            )

        try:
            logger.info("Starting ZIP extraction...")

            with zipfile.ZipFile(self.config.local_data_file, "r") as zip_ref:
                zip_ref.extractall(unzip_path)

            logger.info(
                f"ZIP extracted successfully to: {unzip_path}"
            )

        except Exception as e:
            logger.error(f"Error while extracting ZIP file: {e}")
            raise e

In [10]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-05-13 16:26:41,371: INFO: common : yaml file: config\config.yaml loaded successfully]
[2026-05-13 16:26:41,378: INFO: common : yaml file: params.yaml loaded successfully]
[2026-05-13 16:26:41,381: INFO: common : yaml file: schema.yaml loaded successfully]
[2026-05-13 16:26:41,384: INFO: common : created directory at: artifacts]
[2026-05-13 16:26:41,385: INFO: common : created directory at: artifacts/data_ingestion]
[2026-05-13 16:26:41,387: INFO: 798920940 : Starting file download...]
[2026-05-13 16:26:46,156: INFO: 798920940 : File downloaded successfully!]
[2026-05-13 16:26:46,158: INFO: 798920940 : File location: artifacts/data_ingestion/data.zip]
[2026-05-13 16:26:46,159: INFO: 798920940 : File size: 23329 bytes]
[2026-05-13 16:26:46,174: INFO: 798920940 : Starting ZIP extraction...]
[2026-05-13 16:26:46,183: INFO: 798920940 : ZIP extracted successfully to: artifacts/data_ingestion]
